![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 45 -- Case Study: Haunt Me! (Starter)

The task of this morning's case study, **"Haunt me!"**, from the **Spooky Round 2025** of the Nitro Association
(Romania), held on its judge on 2 November 2025. Try it yourself: the statement, the data loaded and looked at, the score in code, and a first valid file (no ghost anywhere). Then train your own detector; the slides show how to read the task first.

**Sources and credits.** The task, its statement and its data: [Nitro AI Judge, Spooky Round 2025](https://judge.nitro-ai.org/competitions/nitro/spooky-round-2025)
(all rights reserved by the Nitro Association), told here in our words with short quotes. A contestant's solutions
(100 and 90 points): S. Asandei, [roai-solved](https://github.com/stefanasandei/roai-solved)
(`contests/spooky-round-01/haunt-me`, MIT licence). The code of this notebook is the course's own, written for the
Day 45 slides.

**How to run it.** The data is not copied here. Make a free account on the judge, open the task, and download
"Train Data" (`train_data.csv`) and the "Starter Kit" (the photos). Put `train_data.csv` and the folder `images`
(with `images/train` and `images/inference`) in a folder `haunt` next to this notebook (in Colab: upload them, or
mount your Drive and change `DATA_DIR`). A GPU runtime is faster, but the round's rule is a CPU: both work.


## The task, in our words

**The story.** On Halloween night, cameras "capture things that shouldn't exist". Build "a rigorous ghost detector".

**Two subtasks for every photo.**
- **Subtask 1:** is there a ghost? Answer `1` or `0`.
- **Subtask 2:** if there is one, its box `[x_min,y_min,x_max,y_max]` in pixels; if not, `[-1,-1,-1,-1]`.

**The data.** 800 training photos with answers (`train_data.csv`: `image_id, label, bbox_xmin, bbox_ymin, bbox_xmax,
bbox_ymax`; with no ghost, `label` 0 and four `-1`), and 200 photos to answer (`images/inference`). Every photo is
512 x 512.

**The file.** Two rows per photo, 400 rows: `subtaskID,datapointID,answer`, for example
`1,img_0012,0` then `2,img_0012,"[-1,-1,-1,-1]"`. Note: the statement says `[-1,-1,-1,-1]` for no ghost, while the
sample output writes `[0,0,0,0]`; we follow the statement and the training file.

**The score.** Accuracy for subtask 1; "mAP" and "IoU" for subtask 2. The statement does not say how they become
100 points. In this notebook: accuracy, and the mean IoU over all photos, where "no box" on a photo with no ghost
scores 1.

**The rules.** Any model that runs on a CPU in less than 2 hours; pretrained models allowed, but nothing from the
transformer family (and no AI assistant to write the code). YOLOv8 is convolutions only; YOLO11 has attention blocks.

## Setup

In [ ]:
# !pip install ultralytics      # once, on Colab
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

DATA_DIR = "haunt"     # train_data.csv and images/ inside

## Load

In [ ]:
train = pd.read_csv(DATA_DIR + "/train_data.csv")
print(train.shape)
print(train["label"].value_counts())
train.head()

## Look: twelve ghosts, and the shape of every box

In [ ]:
ghosts = train[train["label"] == 1]
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(12):
    row = ghosts.iloc[i]
    image = Image.open(DATA_DIR + "/images/train/" + row["image_id"] + ".jpg")
    w = row["bbox_xmax"] - row["bbox_xmin"]
    h = row["bbox_ymax"] - row["bbox_ymin"]
    box = patches.Rectangle((row["bbox_xmin"], row["bbox_ymin"]), w, h,
                            fill=False, ec="red")
    axes[i // 6, i % 6].imshow(image)
    axes[i // 6, i % 6].add_patch(box)
    axes[i // 6, i % 6].axis("off")
plt.show()

w = ghosts["bbox_xmax"] - ghosts["bbox_xmin"]
h = ghosts["bbox_ymax"] - ghosts["bbox_ymin"]
print("height / width:", (h / w).min(), (h / w).mean(), (h / w).max())

## The score in code, and an answer with no model

In [ ]:
def box_iou(true_box, pred_box):
    # a box is [x1, y1, x2, y2]; None = no box
    if true_box is None and pred_box is None:
        return 1.0
    if true_box is None or pred_box is None:
        return 0.0
    w = min(true_box[2], pred_box[2]) - max(true_box[0], pred_box[0])
    h = min(true_box[3], pred_box[3]) - max(true_box[1], pred_box[1])
    if w <= 0 or h <= 0:
        return 0.0
    area_t = (true_box[2] - true_box[0]) * (true_box[3] - true_box[1])
    area_p = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
    return w * h / (area_t + area_p - w * h)


truth = {}
for i in range(len(train)):
    row = train.iloc[i]
    if row["label"] == 1:
        truth[row["image_id"]] = [row["bbox_xmin"], row["bbox_ymin"], row["bbox_xmax"], row["bbox_ymax"]]
    else:
        truth[row["image_id"]] = None

# two answers that need no model, on all 800 training photos
ids = list(train["image_id"])
no_ghost = []
for image_id in ids:
    no_ghost.append(box_iou(truth[image_id], None))
print("'no ghost' everywhere: accuracy", round((train["label"] == 0).mean(), 3), "mean IoU", round(np.mean(no_ghost), 3))

## The file: two rows per photo

In [ ]:
def to_answer(conf, box, threshold):
    if box is None or conf < threshold:
        return 0, "[-1,-1,-1,-1]"
    x1 = int(round(box[0]))
    y1 = int(round(box[1]))
    x2 = int(round(box[2]))
    y2 = int(round(box[3]))
    text = f"[{x1},{y1},{x2},{y2}]"
    return 1, text

In [ ]:
# a first valid file: "no ghost" for every photo
rows = []
for name in sorted(os.listdir(DATA_DIR + "/images/inference")):
    image_id = name.replace(".jpg", "")
    has_ghost, box_text = to_answer(0.0, None, 0.5)
    rows.append([1, image_id, has_ghost])
    rows.append([2, image_id, box_text])
submission = pd.DataFrame(rows, columns=["subtaskID", "datapointID", "answer"])
submission.to_csv("submission.csv", index=False)
print(len(submission))      # 400
submission.head()

## Your turn

Follow the protocol: read, look, list, a first file, then improve one step at a time. The slides' toolbox shows the
YOLO files (one label file per photo, an empty one when there is no ghost), the training call, the best box per photo
and the threshold chosen on validation.

Made By **Sattam Altwaim**